# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kifayat-Sayed/flyrank-ml-internship1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing the **Content Refresh Opportunity Scoring** lane. The goal is to help content and SEO teams identify which existing pages deserve human review first, instead of relying only on manual checks or fixed rules. The starter dataset contains search performance, engagement, content characteristics, and recent trend information, which gives us several signals to investigate together. I want to understand whether these signals can support a more useful and consistent review-priority ranking. This is a practical problem because teams have limited time and cannot investigate every page at once.

In [1]:
# Define the provisional project lane.

project_lane = "Content Refresh Opportunity Scoring"

project_goal = (
    "Rank existing content pages by review priority "
    "so that human editors can investigate the most relevant pages first."
)

print("Selected lane:", project_lane)
print("Project goal:", project_goal)

Selected lane: Content Refresh Opportunity Scoring
Project goal: Rank existing content pages by review priority so that human editors can investigate the most relevant pages first.


## 2. The question: decision, action, cost of a wrong call

### Research question

**Can existing content and search-performance signals help rank pages by their observed decline status, so that human editors can decide which pages to review first?**

- **Decision:** Which content page should be investigated first?
- **Who acts:** Content editors and SEO teams.
- **Action:** Review the highest-priority pages, investigate possible issues, and decide whether an update is justified.
- **Cost of a wrong recommendation:** A false positive may waste editorial time on a page that does not need attention. A false negative may leave a genuinely declining page unreviewed, potentially delaying corrective action.
- **Why data and ML may help:** Page performance depends on several interacting signals, such as impressions, clicks, engagement, content age, and update history. A transparent rule-based score provides a useful starting point, while a model can be tested to see whether it adds value.

The task is primarily a **ranking problem**. I will evaluate whether the top-ranked pages contain a useful proportion of pages with the observed decline label, using **Precision@K**.

In [2]:
# Record the research question as a structured problem frame.

research_frame = {
    "decision": "Which content pages should be reviewed first?",
    "user": "Content editors and SEO teams",
    "action": "Investigate high-priority pages and decide whether to refresh them",
    "task_type": "Ranking / scoring",
    "target": "Observed decline status",
    "primary_metric": "Precision@K",
    "false_positive_cost": "Editorial time spent reviewing a page that does not need attention",
    "false_negative_cost": "A declining page may remain unreviewed",
    "baseline": "A transparent rule-based priority score"
}

for key, value in research_frame.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

# Check that the essential parts of the problem are defined.
required_fields = ["decision", "user", "action", "task_type", "target", "primary_metric"]

assert all(research_frame[field] for field in required_fields)

print("\nProblem framing check: passed")

Decision: Which content pages should be reviewed first?
User: Content editors and SEO teams
Action: Investigate high-priority pages and decide whether to refresh them
Task Type: Ranking / scoring
Target: Observed decline status
Primary Metric: Precision@K
False Positive Cost: Editorial time spent reviewing a page that does not need attention
False Negative Cost: A declining page may remain unreviewed
Baseline: A transparent rule-based priority score

Problem framing check: passed


## 3. Quick look at the data (2-3 real numbers)

### Initial data observations

The project uses the anonymized `content_refresh_anonymized.csv` starter dataset. It contains **30,000 content-page records**, **44 columns**, and data from **32 pseudonymized clients**. The records contain a trailing 90-day performance snapshot, including search impressions, clicks, sessions, engagement, content age, and recent performance trends.

One useful starting observation is that approximately **54.2% of the records have the observed decline label**. This gives us a substantial group of declining pages to investigate. However, the label is derived from the dataset's `trend_direction` field, so it must be treated as an observed dataset label—not as independent proof of future decline.

The initial figures justify further investigation, but they do not establish that a machine-learning model will outperform a simple rule-based approach.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the anonymized starter dataset and inspect basic statistics.

from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "Kifayat-Sayed/flyrank-ml-internship1/"
    "main/data/raw/content_refresh_anonymized.csv"
)

# Use a local copy if available; otherwise load from the public repository.
if DATA_PATH.exists():
    data = pd.read_csv(DATA_PATH)
    data_source = "Local repository copy"
else:
    data = pd.read_csv(DATA_URL)
    data_source = "Public GitHub repository"

# Confirm that the columns needed for this initial inspection exist.
required_columns = ["client_id", "trend_direction", "content_id"]

missing_columns = [
    column for column in required_columns
    if column not in data.columns
]

assert not missing_columns, f"Missing required columns: {missing_columns}"

# Calculate aggregate statistics.
total_records = len(data)
total_columns = len(data.columns)
unique_clients = data["client_id"].nunique()

declining_records = (
    data["trend_direction"]
    .astype("string")
    .str.lower()
    .eq("down")
    .sum()
)

decline_percentage = (declining_records / total_records) * 100

# Display only aggregate values, not individual client or content identifiers.
print("DATASET QUICK CHECK")
print("-" * 35)
print("Data source:", data_source)
print(f"Content records: {total_records:,}")
print(f"Columns: {total_columns}")
print(f"Pseudonymized clients: {unique_clients}")
print(f"Records labelled as declining: {declining_records:,}")
print(f"Declining records: {decline_percentage:.2f}%")

# Basic integrity checks.
assert total_records == 30000
assert unique_clients == 32
assert total_columns == 44
assert 0 <= decline_percentage <= 100

print("\nDataset inspection: passed")

DATASET QUICK CHECK
-----------------------------------
Data source: Public GitHub repository
Content records: 30,000
Columns: 44
Pseudonymized clients: 32
Records labelled as declining: 16,262
Declining records: 54.21%

Dataset inspection: passed


## 4. Careful words: what I can and can't claim
### Scope of claims

This project will make **observed, measured, directional, and decision-support** claims.

I can investigate relationships between the available content and performance signals and the dataset's observed decline label. I can compare a transparent rule-based baseline with machine-learning approaches and measure how effectively they rank pages for human review.

There are important limitations:

- The dataset is a snapshot containing aggregated performance measurements, not a controlled experiment.
- The decline label is derived from `trend_direction`, which is itself based on `trend_pct`. These fields must not be used as model input features when predicting the label.
- The available data does not prove that refreshing a page will improve its search performance.
- The analysis cannot establish that Google will change a page's ranking or predict Google's ranking algorithm.
- Any model output should be treated as a review-priority recommendation for a human, not an automatic instruction to modify content.

The final conclusions will depend on validation results and comparison with a rule-based baseline. If the model does not outperform the baseline under an appropriate evaluation split, that result will be reported honestly.

In [4]:


allowed_claims = [
    "Describe observed patterns in the available dataset",
    "Measure performance using a defined evaluation metric",
    "Compare a machine-learning model with a rule-based baseline",
    "Rank pages for human review as decision support"
]

unsupported_claims = [
    "Prove that refreshing content causes performance to improve",
    "Predict Google's ranking algorithm",
    "Guarantee that a recommended page will recover",
    "Treat the observed decline label as a future outcome"
]

# Record known label-related leakage risks.
label_source = "trend_direction"
related_trend_field = "trend_pct"

prohibited_features = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

assert label_source in prohibited_features
assert related_trend_field in prohibited_features
assert "content_id" in prohibited_features
assert "client_id" in prohibited_features

print("CLAIM BOUNDARIES")
print("-" * 35)

print("\nClaims this project may support:")
for claim in allowed_claims:
    print("-", claim)

print("\nClaims this project cannot establish:")
for claim in unsupported_claims:
    print("-", claim)

print("\nFields excluded from model features:")
for field in prohibited_features:
    print("-", field)

print("\nClaim-scope and leakage-awareness check: passed")

CLAIM BOUNDARIES
-----------------------------------

Claims this project may support:
- Describe observed patterns in the available dataset
- Measure performance using a defined evaluation metric
- Compare a machine-learning model with a rule-based baseline
- Rank pages for human review as decision support

Claims this project cannot establish:
- Prove that refreshing content causes performance to improve
- Predict Google's ranking algorithm
- Guarantee that a recommended page will recover
- Treat the observed decline label as a future outcome

Fields excluded from model features:
- trend_direction
- trend_pct
- content_id
- client_id

Claim-scope and leakage-awareness check: passed


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.